# RCC Subtype Classification - Feature Extraction
### BME 515 Final Project
Reads patches from patches/, extracts ResNet50 features, saves as .h5 files in features/

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from pathlib import Path
import numpy as np
import h5py

BASE_DIR = Path('/Users/zhangruikun/Desktop/BME515')
PATCH_DIR = BASE_DIR / 'patches'
FEATURE_DIR = BASE_DIR / 'features'
FEATURE_DIR.mkdir(exist_ok=True)

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'Patch dir exists: {PATCH_DIR.exists()}')
print(f'Feature dir exists: {FEATURE_DIR.exists()}')

In [ ]:
# load ResNet50, strip final classification layer
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
model = torch.nn.Sequential(*list(model.children())[:-1])
model = model.to(device)
model.eval()
print('Model ready — output: 2048-dim feature vector per patch')

In [ ]:
# ImageNet normalization (required for ResNet50)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
def parse_coords(patch_path):
    # parse y, x from filename: DHMC_0001_y256_x512.png -> (512, 256)
    parts = patch_path.stem.split('_')
    y = int(parts[-2][1:])
    x = int(parts[-1][1:])
    return x, y  # CLAM expects (x, y)

def extract_slide_features(slide_name, batch_size=32):
    patch_paths = sorted((PATCH_DIR / slide_name).glob('*.png'))
    if not patch_paths:
        print(f'No patches found: {slide_name}')
        return None, None

    all_features, all_coords = [], []

    with torch.no_grad():
        for i in range(0, len(patch_paths), batch_size):
            batch_paths = patch_paths[i:i+batch_size]
            batch = [transform(Image.open(p).convert('RGB')) for p in batch_paths]
            coords = [parse_coords(p) for p in batch_paths]

            feats = model(torch.stack(batch).to(device)).squeeze(-1).squeeze(-1)
            all_features.append(feats.cpu())
            all_coords.extend(coords)

    return (
        torch.cat(all_features, dim=0).numpy(),  # [N, 2048]
        np.array(all_coords, dtype=np.int32)      # [N, 2]
    )

In [ ]:
# test on first slide before running all
test_slide = 'DHMC_0001'
feats, coords = extract_slide_features(test_slide)
print(f'{test_slide}: features {feats.shape}, coords {coords.shape}')

with h5py.File(FEATURE_DIR / f'{test_slide}.h5', 'w') as f:
    f.create_dataset('features', data=feats)
    f.create_dataset('coords', data=coords)
print(f'Saved {test_slide}.h5 — looks good, ready to run all')

In [ ]:
# run all slides in patches/
slide_dirs = sorted([d for d in PATCH_DIR.iterdir() if d.is_dir()])
print(f'Found {len(slide_dirs)} slides')

for i, slide_dir in enumerate(slide_dirs):
    slide_name = slide_dir.name
    save_path = FEATURE_DIR / f'{slide_name}.h5'

    if save_path.exists():
        continue  # skip if already done

    feats, coords = extract_slide_features(slide_name)
    if feats is not None:
        with h5py.File(save_path, 'w') as f:
            f.create_dataset('features', data=feats)
            f.create_dataset('coords', data=coords)

    if (i + 1) % 10 == 0:
        print(f'[{i+1}/{len(slide_dirs)}] done')

print(f'Complete. {len(list(FEATURE_DIR.glob("*.h5")))} .h5 files in features/')

In [ ]:
# verify
h5_files = sorted(FEATURE_DIR.glob('*.h5'))
print(f'Total .h5 files: {len(h5_files)}')
with h5py.File(h5_files[0], 'r') as f:
    print(f'Sample: {h5_files[0].name}')
    print(f'  features: {f["features"].shape}')
    print(f'  coords:   {f["coords"].shape}')